In [51]:
import numpy as np

from cryosiam.transforms.array import RandomMaskedViews
from cryosiam.networks.nets import DenseSimSiam

In [53]:
patch1 = np.array([[[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12], [13, 14, 15, 16]]])
patch2 = np.array([[[17, 18, 19, 20], [21, 22, 23, 24], [25, 26, 27, 28], [29, 30, 31, 32]]])

In [54]:
patch1.shape

(1, 4, 4)

In [55]:
view_finder = RandomMaskedViews(patch1.shape[1:], view_size=(2,2), overlap=0.5)

In [56]:
views1 = view_finder(patch1)
views2 = view_finder(patch2)

In [57]:
print(views1['view1'][0])
print(views1['view2'][0])

[[3 4]
 [7 8]]
[[ 7  8]
 [11 12]]


In [58]:
print(patch1)

[[[ 1  2  3  4]
  [ 5  6  7  8]
  [ 9 10 11 12]
  [13 14 15 16]]]


In [77]:
print(views2['view1'][0])
print(views2['view2'][0])

[[26 27]
 [30 31]]
[[25 26]
 [29 30]]


In [78]:
print(patch2)

[[[17 18 19 20]
  [21 22 23 24]
  [25 26 27 28]
  [29 30 31 32]]]


In [59]:
import torch

def select_overlap_pixels(feats, mask):
        shape = feats.size()
        masked = None
        for i in range(shape[0]):
            current_mask = mask[i]
            if masked is None:
                if len(shape) > 4:
                    shape_mask = (shape[0], shape[1],
                                  current_mask[0][1] - current_mask[0][0] + 1,
                                  current_mask[1][1] - current_mask[1][0] + 1,
                                  current_mask[2][1] - current_mask[2][0] + 1)
                else:
                    shape_mask = (shape[0], shape[1],
                                  current_mask[0][1] - current_mask[0][0] + 1,
                                  current_mask[1][1] - current_mask[1][0] + 1)
                masked = torch.zeros(shape_mask, dtype=torch.float)
                masked = masked.to(feats.get_device())
            if len(shape) > 4:
                masked[i, :, :, :, :] = feats[i, :, current_mask[0][0]:current_mask[0][1] + 1,
                                        current_mask[1][0]:current_mask[1][1] + 1,
                                        current_mask[2][0]:current_mask[2][1] + 1]
            else:
                masked[i, :, :, :] = feats[i, :, current_mask[0][0]:current_mask[0][1] + 1,
                                     current_mask[1][0]:current_mask[1][1] + 1]
        return masked

In [61]:
patch_cmb = np.stack((patch1, patch2), axis=0)
print(patch_cmb.shape)

(2, 1, 4, 4)


In [62]:
views1['view1'].shape

(1, 2, 2)

In [64]:
# combine view1 into a single array
view1_cmb = np.stack((views1['view1'], views2['view1']), axis=0)
view2_cmb = np.stack((views1['view2'], views2['view2']), axis=0)
print(view1_cmb.shape)
print(view2_cmb.shape)

(2, 1, 2, 2)
(2, 1, 2, 2)


In [65]:
# Combine masks into a single array
mask1_cmb = np.stack((views1['mask1'], views2['mask1']), axis=0)
print(mask1_cmb.shape)  
mask2_cmb = np.stack((views1['mask2'], views2['mask2']), axis=0)
print(mask2_cmb.shape)

(2, 2, 2)
(2, 2, 2)


In [81]:
view1_cmb[0, 0, 1:2, 0:2].shape

(1, 2)

In [80]:
view1_cmb[1, 0, 0:2, 0:1].shape

(2, 1)

In [70]:
view1_tensor = torch.from_numpy(view1_cmb)
#send view 1 to cpu device
view1_tensor = view1_tensor.to('cuda:6')

In [71]:
view1_tensor.get_device()

6

In [76]:
overlap = select_overlap_pixels(view1_tensor, mask1_cmb)

RuntimeError: The expanded size of the tensor (1) must match the existing size (2) at non-singleton dimension 1.  Target sizes: [1, 1, 2].  Tensor sizes: [2, 1]